<a href="https://colab.research.google.com/github/NavneethRamesh/SIH2026-TRACE/blob/main/LeadAcetate_H2S_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

def clean_h2s_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms raw gas test data into standardized metrics based on the
    Bureau of Standards technologic paper No. 41[cite: 1].
    """
    # 1. Temperature Normalization
    # Convert mixing reservoir temps (e.g., 27C) to Fahrenheit for consistency[cite: 2].
    if 'temp_celsius' in df.columns:
        df['temp_fahrenheit'] = df['temp_celsius'] * 9/5 + 32

    # 2. Color Metric Mapping
    # 1 = Blank, 10 = Very dark brown[cite: 4].
    color_map = {
        'blank': 1,
        'faint': 2,
        'distinct': 5,
        'dark_brown': 8,
        'very_dark_brown': 10
    }
    df['color_score'] = df['color_observation'].map(color_map).fillna(1)

    # 3. Concentration Checks
    # Recommended lead acetate solution is 5%[cite: 1].
    df['is_optimal_solution'] = np.where(df['pb_acetate_pct'] == 5.0, True, False)

    # Flag gas containing non-permissible amounts (> 0.5 grains per 100 cu ft)
    # A 1-minute test easily detects 0.5 grains[cite: 1].
    df['fails_inspection'] = np.where(df['h2s_grains_per_100cf'] >= 0.5, True, False)

    return df

# Example instantiation:
# raw_data = pd.DataFrame({
#     'temp_celsius': [27, 22],
#     'color_observation': ['faint', 'very_dark_brown'],
#     'pb_acetate_pct': [5.0, 6.5],
#     'h2s_grains_per_100cf': [0.17, 0.72]
# })
# processed_pipeline = clean_h2s_data(raw_data)